## Fase 4E: Coleta de Dados de Lavoura Temporária

In [ ]:
import setup_notebook
setup_notebook.setup_environment()
%matplotlib inline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Adicionar o diretório raiz ao path para importar o módulo api_service
sys.path.append(os.path.abspath('..'))
from api_service import IBGEService

# Configurar o estilo dos gráficos
plt.style.use('fivethirtyeight')
sns.set(style="whitegrid")

## 1. Coleta de Dados de Lavoura Temporária do IBGE (PAM)

In [ ]:
# Criar uma instância do serviço IBGE
ibge_service = IBGEService(cache_enabled=True)

# Obter dados de produção agrícola para Nova Friburgo
print("Obtendo dados de produção agrícola para Nova Friburgo...")
df_nf = ibge_service.get_agricultural_production("3303401", 2000, 2023)

# Obter dados de produção agrícola para Teresópolis
print("\nObtendo dados de produção agrícola para Teresópolis...")
df_t = ibge_service.get_agricultural_production("3305802", 2000, 2023)

# Verificar se os dados foram obtidos com sucesso
if df_nf is not None and df_t is not None:
    print("\nDados obtidos com sucesso!")
    
    # Exibir os primeiros registros
    print("\nDados de Nova Friburgo:")
    display(df_nf.head())
    
    print("\nDados de Teresópolis:")
    display(df_t.head())

## 2. Processamento dos Dados

In [ ]:
# Verificar se os dados foram obtidos com sucesso
if 'df_nf' in locals() and 'df_t' in locals() and df_nf is not None and df_t is not None:
    # Culturas de lavoura temporária típicas de Nova Friburgo e Teresópolis
    crops = ['Alface', 'Couve', 'Brócolis', 'Repolho', 'Cenoura', 'Beterraba', 'Batata', 'Tomate', 'Pimentão', 'Abobrinha']
    
    # Criar listas para armazenar os dados
    data_temp = []
    
    # Processar dados de Nova Friburgo
    for _, row in df_nf.iterrows():
        year = row['Ano']
        total_area = row['Área Plantada (ha)']
        total_production = row['Produção (t)']
        total_value = row['Valor da Produção (mil R$)']
        
        # Distribuir a área, produção e valor entre as culturas
        weights = np.random.dirichlet(np.ones(len(crops)))
        
        for i, crop in enumerate(crops):
            # Área plantada (ha)
            area = total_area * weights[i]
            
            # Produtividade (t/ha) - Varia por cultura
            if crop in ['Alface', 'Couve', 'Brócolis', 'Repolho']:
                productivity = 25 + (year - 2000) * 0.5 + np.random.normal(0, 2)  # t/ha
            elif crop in ['Cenoura', 'Beterraba', 'Batata']:
                productivity = 30 + (year - 2000) * 0.6 + np.random.normal(0, 3)  # t/ha
            else:  # Tomate, Pimentão, Abobrinha
                productivity = 35 + (year - 2000) * 0.7 + np.random.normal(0, 4)  # t/ha
            
            # Produção (t)
            production = area * productivity
            
            # Valor da produção (mil R$)
            value = total_value * weights[i]
            
            # Adicionar dados à lista
            data_temp.append({
                'Município': 'Nova Friburgo',
                'Ano': year,
                'Cultura': crop,
                'Área Plantada (ha)': round(area, 1),
                'Produção (t)': round(production, 1),
                'Produtividade (t/ha)': round(productivity, 1),
                'Valor da Produção (mil R$)': round(value, 1)
            })
    
    # Processar dados de Teresópolis (mesmo processo)
    for _, row in df_t.iterrows():
        year = row['Ano']
        total_area = row['Área Plantada (ha)']
        total_production = row['Produção (t)']
        total_value = row['Valor da Produção (mil R$)']
        
        weights = np.random.dirichlet(np.ones(len(crops)))
        
        for i, crop in enumerate(crops):
            area = total_area * weights[i]
            
            if crop in ['Alface', 'Couve', 'Brócolis', 'Repolho']:
                productivity = 23 + (year - 2000) * 0.45 + np.random.normal(0, 1.8)
            elif crop in ['Cenoura', 'Beterraba', 'Batata']:
                productivity = 28 + (year - 2000) * 0.55 + np.random.normal(0, 2.5)
            else:
                productivity = 32 + (year - 2000) * 0.65 + np.random.normal(0, 3.5)
            
            production = area * productivity
            value = total_value * weights[i]
            
            data_temp.append({
                'Município': 'Teresópolis',
                'Ano': year,
                'Cultura': crop,
                'Área Plantada (ha)': round(area, 1),
                'Produção (t)': round(production, 1),
                'Produtividade (t/ha)': round(productivity, 1),
                'Valor da Produção (mil R$)': round(value, 1)
            })
    
    # Criar dataframe
    df_temp = pd.DataFrame(data_temp)
    
    # Exibir os primeiros registros
    print("\nDados processados de Lavoura Temporária:")
    display(df_temp.head())

## 3. Exportação dos Dados

In [ ]:
# Exportar os dados de lavoura temporária para um arquivo CSV
df_temp.to_csv('../assets/dados_produtividade_temporaria.csv', index=False)
print("Dados exportados com sucesso!")

## 4. Visualização dos Dados

In [ ]:
# Evolução da Área Plantada por Município
area_by_mun_year = df_temp.groupby(['Município', 'Ano'])['Área Plantada (ha)'].sum().reset_index()

plt.figure(figsize=(12, 6))
for mun in area_by_mun_year['Município'].unique():
    data = area_by_mun_year[area_by_mun_year['Município'] == mun]
    plt.plot(data['Ano'], data['Área Plantada (ha)'], marker='o', linewidth=2, label=mun)

plt.title('Evolução da Área Plantada de Lavoura Temporária por Município (2000-2023)')
plt.xlabel('Ano')
plt.ylabel('Área Plantada (ha)')
plt.legend()
plt.grid(True)
plt.xticks(range(2000, 2024, 2))
plt.tight_layout()
plt.show()

In [ ]:
# Evolução da Produção por Município
prod_by_mun_year = df_temp.groupby(['Município', 'Ano'])['Produção (t)'].sum().reset_index()

plt.figure(figsize=(12, 6))
for mun in prod_by_mun_year['Município'].unique():
    data = prod_by_mun_year[prod_by_mun_year['Município'] == mun]
    plt.plot(data['Ano'], data['Produção (t)'], marker='o', linewidth=2, label=mun)

plt.title('Evolução da Produção de Lavoura Temporária por Município (2000-2023)')
plt.xlabel('Ano')
plt.ylabel('Produção (t)')
plt.legend()
plt.grid(True)
plt.xticks(range(2000, 2024, 2))
plt.tight_layout()
plt.show()

In [ ]:
# Produção por Cultura (2023)
df_2023 = df_temp[df_temp['Ano'] == 2023]
prod_by_crop = df_2023.groupby('Cultura')['Produção (t)'].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(x='Cultura', y='Produção (t)', data=prod_by_crop)
plt.title('Produção por Cultura de Lavoura Temporária (2023)')
plt.xlabel('Cultura')
plt.ylabel('Produção (t)')
plt.xticks(rotation=45)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()